<a href="https://colab.research.google.com/github/Guille-1905/Analisis_resultados_UNAM_2018-2026/blob/main/1_Preparaci%C3%B3n_y_validaci%C3%B3n_de_datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Preparación y validación de datos.
##Objetivo
El objetivo de este notebook es preparar las bases de datos utilizadas en el estudio mediante un proceso sistemático de auditoría, corrección, limpieza y validación. Esta etapa busca garantizar la integridad, consistencia y calidad de la información antes de realizar el análisis estadístico, asegurando además que todas las transformaciones aplicadas sean documentadas y reproducibles.

----


In [ ]:
#@title Cargar librerías. {display-mode: 'form'}
import pandas as pd
import numpy as np

In [ ]:
#@title Cargar datos. {display-mode: 'form'}

try:
  df_aspirantes = pd.read_parquet('https://github.com/Guille-1905/An-lisis_Resultados_UNAM_2026/raw/refs/heads/main/estadisticas_alumnos_UNAM.parquet')
  df_carreras = pd.read_parquet('https://github.com/Guille-1905/An-lisis_Resultados_UNAM_2026/raw/refs/heads/main/estadisticas_carreras_UNAM.parquet')

  print("Las bases han sido cargadas con éxito")
except Exception as e:
  print(f"Error al cargar las bases: {e}")

Las bases han sido cargadas con éxito


##1.1 Dataset `estadisticas_aspirantes_UNAM`
###1.1.1 Auditoria general


In [ ]:
#@title Dimensión

df_aspirantes.shape

(1705246, 8)

In [ ]:
#@title Registros

df_aspirantes.head()

,año,concurso,area,carrera,plantel,comprobante,aciertos,seleccionado
0,2018,Febrero,1,ACTUARIA,FACULTAD DE CIENCIAS,000017,82.0,No
1,2018,Febrero,1,ACTUARIA,FACULTAD DE CIENCIAS,000043,109.0,Sí
2,2018,Febrero,1,ACTUARIA,FACULTAD DE CIENCIAS,000059,106.0,Sí
3,2018,Febrero,1,ACTUARIA,FACULTAD DE CIENCIAS,000070,96.0,No
4,2018,Febrero,1,ACTUARIA,FACULTAD DE CIENCIAS,000073,62.0,No


In [ ]:
#@title Estructura de las variables

df_aspirantes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1705246 entries, 0 to 1705245
Data columns (total 8 columns):
 #   Column        Dtype  
---  ------        -----  
 0   año           int64  
 1   concurso      object 
 2   area          object 
 3   carrera       object 
 4   plantel       object 
 5   comprobante   object 
 6   aciertos      float64
 7   seleccionado  object 
dtypes: float64(1), int64(1), object(6)
memory usage: 104.1+ MB


In [ ]:
#@title Descripción de las variables numéricas

df_aspirantes.describe()

,año,aciertos
count,1.705246e+06,1.481624e+06
mean,2.021703e+03,5.799488e+01
std,2.612088e+00,1.991357e+01
min,2.018000e+03,0.000000e+00
25%,2.019000e+03,4.300000e+01
50%,2.022000e+03,5.400000e+01
75%,2.024000e+03,7.000000e+01
max,2.026000e+03,1.200000e+02


In [ ]:
#@title Valores faltantes

df_aspirantes.isna().sum()

,0
año,0
concurso,0
area,0
carrera,0
plantel,0
comprobante,0
aciertos,223622
seleccionado,0


In [ ]:
#@title Valores duplicados

df_aspirantes.duplicated().sum()

np.int64(0)

Después de analizar de manera ganeral nuestro conjunto de datos podemos observar que la tabla cuenta con un total de 1,705,246 registros almacenados en 8 variables (columnas). De estas 8 variables, podemos observar que la clave única de registro depende de tres de ellas (`año`, `concurso`, `comprobante`), mientras que el resto almacenan información como el catálogo de carreras ofertadas por la UNAM (`carrera`) junto con el catálogo de planteles (`plantel`) en los que se imparten dichas carreras.
Las variables `concurso`, `area`, `carrera`, `plantel`, `comprobante` y `seleccionado` se encuentran almacenadas con el tipo `object`. Debido a su naturaleza categórica o textual, posteriormente se evaluará su conversión a tipos de datos más adecuados (`category` o `string`) para optimizar el procesamiento y garantizar la consistencia de la información.

Por otro lado, la única variable que presenta información faltante es aciertos, con 223,622 valores ausentes. Debido a la importancia de esta variable para el análisis posterior, será necesario investigar el origen de dichas ausencias antes de definir una estrategia de tratamiento.

###1.1.2 Correción de errores identificados

In [ ]:
#@title Estandarización de variables

df_aspirantes[['concurso', 'seleccionado']] = df_aspirantes[['concurso', 'seleccionado']].astype('category')
df_aspirantes['area'] = pd.to_numeric(df_aspirantes['area'], errors='raise').astype('Int64').astype('category')
df_aspirantes['carrera'] = df_aspirantes['carrera'].astype('string')
df_aspirantes['plantel'] = df_aspirantes['plantel'].astype('string')
df_aspirantes['comprobante'] = df_aspirantes['comprobante'].astype('string')

df_aspirantes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1705246 entries, 0 to 1705245
Data columns (total 8 columns):
 #   Column        Dtype   
---  ------        -----   
 0   año           int64   
 1   concurso      category
 2   area          category
 3   carrera       string  
 4   plantel       string  
 5   comprobante   string  
 6   aciertos      float64 
 7   seleccionado  category
dtypes: category(3), float64(1), int64(1), string(3)
memory usage: 69.9 MB


Con base en los resultados obtenidos mediante la auditoria general, se estandarizaron los tipos de datos de las variables para que fueran consistentes con las infromación que almacenan. En particular se almacenaron como `category` las variables `concurso`, `area` y `seleccionado`, mientras que `carrera`, `plantel` y `comprobante` se almacenaron como `string`.

In [ ]:
#@title Estandarización de catálogos (`carrera` y `plantel`)

catalogo_carreras = pd.read_excel("/content/catalogo_carreras_corregido.xlsx")
catalogo_planteles = pd.read_excel("/content/catalogo_planteles_corregido.xlsx")
catalogo_unificacion = pd.read_excel("/content/carreras_unificadas.xlsx")

diccionario_carreras = dict(zip(catalogo_carreras["carrera"], catalogo_carreras["carrera_corregido"]))
diccionario_planteles = dict(zip(catalogo_planteles["plantel"], catalogo_planteles["plantel_corregido"]))

df_carreras_actualizado = df_carreras.copy()
df_aspirantes_actualizado = df_aspirantes.copy()

df_carreras_actualizado["carrera"] = (df_carreras_actualizado["carrera"].replace(diccionario_carreras))
df_aspirantes_actualizado["carrera"] = (df_aspirantes_actualizado["carrera"].replace(diccionario_carreras))

df_carreras_actualizado["plantel"] = (df_carreras_actualizado["plantel"].replace(diccionario_planteles))
df_aspirantes_actualizado["plantel"] = (df_aspirantes_actualizado["plantel"].replace(diccionario_planteles))

plantel_antiguo = "ESCUELA NAL. DE ENFERMERIA Y OBSTETRICIA"
plantel_nuevo = "FACULTAD DE ENFERMERIA Y OBSTETRICIA"

df_carreras_actualizado.loc[df_carreras_actualizado["plantel"].eq(plantel_antiguo),"plantel"] = plantel_nuevo
df_aspirantes_actualizado.loc[df_aspirantes_actualizado["plantel"].eq(plantel_antiguo),"plantel"] = plantel_nuevo

criterio_ciencias_tierra_carreras = (df_carreras_actualizado["carrera"].eq("CIENCIAS DE LA TIERRA")& df_carreras_actualizado["plantel"].eq("FACULTAD DE CIENCIAS"))
df_carreras_actualizado.loc[criterio_ciencias_tierra_carreras,"plantel"] = "ESCUELA NAL. DE CIENCIAS DE LA TIERRA"

criterio_ciencias_tierra_aspirantes = (df_aspirantes_actualizado["carrera"].eq("CIENCIAS DE LA TIERRA")& df_aspirantes_actualizado["plantel"].eq("FACULTAD DE CIENCIAS"))
df_aspirantes_actualizado.loc[criterio_ciencias_tierra_aspirantes,"plantel"] = "ESCUELA NAL. DE CIENCIAS DE LA TIERRA"

df_carreras = df_carreras_actualizado.copy()
df_aspirantes = df_aspirantes_actualizado.copy()

##1.2 Dataset `estadisticas_carreras_UNAM`
###1.2.1 Auditoria general

In [ ]:
#@title Dimensión

df_carreras.shape

(1958, 11)

In [ ]:
#@title Registros

df_carreras.head()

,año,concurso,area,carrera,plantel,oferta,aspirantes,presentaron_examen,aciertos_minimos,seleccionados,url
0,2018,Febrero,1,ACTUARIA,FACULTAD DE CIENCIAS,40,1431,1324,106,43,https://www.dgae.unam.mx/Febrero2018/resultado...
1,2018,Febrero,1,ACTUARIA,FES ACATLAN,36,690,642,95,36,https://www.dgae.unam.mx/Febrero2018/resultado...
2,2018,Febrero,1,ARQUITECTURA,FACULTAD DE ARQUITECTURA,190,3790,3516,92,204,https://www.dgae.unam.mx/Febrero2018/resultado...
3,2018,Febrero,1,ARQUITECTURA,FES ACATLAN,62,1073,992,79,68,https://www.dgae.unam.mx/Febrero2018/resultado...
4,2018,Febrero,1,ARQUITECTURA,FES ARAGON,60,1366,1295,79,63,https://www.dgae.unam.mx/Febrero2018/resultado...


In [ ]:
#@title Estructura de las variables

df_carreras.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1958 entries, 0 to 1957
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   año                 1958 non-null   Int64 
 1   concurso            1958 non-null   object
 2   area                1958 non-null   Int64 
 3   carrera             1958 non-null   object
 4   plantel             1958 non-null   object
 5   oferta              1958 non-null   Int64 
 6   aspirantes          1958 non-null   Int64 
 7   presentaron_examen  1958 non-null   Int64 
 8   aciertos_minimos    1958 non-null   Int64 
 9   seleccionados       1958 non-null   Int64 
 10  url                 1958 non-null   object
dtypes: Int64(7), object(4)
memory usage: 181.8+ KB


In [ ]:
#@title Descripción de las variables numéricas

df_carreras.describe()

,año,area,oferta,aspirantes,presentaron_examen,aciertos_minimos,seleccionados
count,1958.0,1958.0,1958.0,1958.0,1958.0,1958.0,1958.0
mean,2021.39428,2.466803,60.775281,870.912155,760.730337,75.998468,68.050562
std,2.657373,1.15426,73.957773,1460.510095,1261.724733,20.615652,85.358877
min,2018.0,1.0,2.0,2.0,1.0,40.0,0.0
25%,2019.0,1.0,20.0,100.0,81.0,60.0,20.0
50%,2021.0,2.0,36.0,385.0,328.0,78.0,38.0
75%,2024.0,4.0,75.0,1077.25,947.0,93.0,84.75
max,2026.0,4.0,586.0,16507.0,13941.0,117.0,613.0


In [ ]:
#@title Valores faltantes

df_carreras.isna().sum()

,0
año,0
concurso,0
area,0
carrera,0
plantel,0
oferta,0
aspirantes,0
presentaron_examen,0
aciertos_minimos,0
seleccionados,0


In [ ]:
#@title Valores duplicados

df_carreras.duplicated().sum()

np.int64(0)

Después de esta auditoria general podemos observar que la tabla contiene 1,958 registros almacenados en 11 variables. Cada registro resume la información estadística correspondiente a una combinación de `año`, `concurso`, `carrera` y `plantel`, incluyendo indicadores como la oferta disponible, el número de aspirantes, quienes presentaron el examen, el puntaje mínimo de ingreso y el número de seleccionados. Asimismo, la auditoría no identificó valores faltantes ni registros duplicados, lo que indica que la base presenta una estructura íntegra antes de iniciar la etapa de preparación.

Las variables `concurso`, `carrera`, `plantel` y `url` se almacenan como tipo `object`, mientras que `area` en tipo `int64`. Por lo tanto, se estandarizarán a `category` las variables `area` y `concurso`, mientras que `carrera` y `plantel` en `string`. Por otro lado la variable `url` corresponde a la dirección de la página de origen de cada registro. Dado que no participa en las etapas de integración ni en los análisis estadísticos del proyecto, se considera prescindible y será eliminada durante la preparación de los datos.

###1.2.2 Corrección de errores identificados

In [ ]:
#@title Estandarización de variables

df_carreras[['area', 'concurso']] = df_carreras[['area', 'concurso']].astype('category')
df_carreras['carrera'] = df_carreras['carrera'].astype('string')
df_carreras['plantel'] = df_carreras['plantel'].astype('string')
df_carreras['año'] = df_carreras['año'].astype('int64')

df_carreras.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1958 entries, 0 to 1957
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   año                 1958 non-null   int64   
 1   concurso            1958 non-null   category
 2   area                1958 non-null   category
 3   carrera             1958 non-null   string  
 4   plantel             1958 non-null   string  
 5   oferta              1958 non-null   Int64   
 6   aspirantes          1958 non-null   Int64   
 7   presentaron_examen  1958 non-null   Int64   
 8   aciertos_minimos    1958 non-null   Int64   
 9   seleccionados       1958 non-null   Int64   
 10  url                 1958 non-null   object  
dtypes: Int64(5), category(2), int64(1), object(1), string(2)
memory usage: 151.5+ KB


In [ ]:
#@title Exclusión de la variable `url`

df_carreras = df_carreras.drop('url', axis=1)
df_carreras.head()

,año,concurso,area,carrera,plantel,oferta,aspirantes,presentaron_examen,aciertos_minimos,seleccionados
0,2018,Febrero,1,ACTUARIA,FACULTAD DE CIENCIAS,40,1431,1324,106,43
1,2018,Febrero,1,ACTUARIA,FES ACATLAN,36,690,642,95,36
2,2018,Febrero,1,ARQUITECTURA,FACULTAD DE ARQUITECTURA,190,3790,3516,92,204
3,2018,Febrero,1,ARQUITECTURA,FES ACATLAN,62,1073,992,79,68
4,2018,Febrero,1,ARQUITECTURA,FES ARAGON,60,1366,1295,79,63


----
##1.3 Auditoria de consitencia entre bases
En esta sección se evaluará la consistencia entre ambas bases de datos. En particular, se verificará que las variables utilizadas para relacionarlas (`año`, `concurso`, `area`, `carrera` y `plantel`) compartan el mismo dominio de valores y que la información estadística común entre ambas fuentes presente un comportamiento consistente. Esta verificación permitirá determinar si ambas bases pueden integrarse de manera confiable para el análisis posterior.
###1.3.1 Consistencia entre catálogos
La primera etapa de la auditoría consiste en comparar los catálogos de las variables compartidas por ambas bases. El objetivo es verificar que los valores utilizados para representar la misma información sean consistentes, ya que cualquier diferencia en la nomenclatura impediría una correcta integración de los datos.

In [ ]:
#@title Variable `año`

año_aspirantes = set(df_aspirantes['año'])
año_carreras = set(df_carreras['año'])

print(año_aspirantes)
print(año_carreras)

{2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026}
{2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026}


In [ ]:
#@title Variable `concurso`

concurso_aspirantes = set(df_aspirantes['concurso'])
concurso_carreras = set(df_carreras['concurso'])

print(concurso_aspirantes)
print(concurso_carreras)

{'Febrero', 'Mayo', 'Junio'}
{'Febrero', 'Mayo', 'Junio'}


In [ ]:
#@title Variable `area`

area_aspirantes = set(df_aspirantes['area'])
area_carreras = set(df_carreras['area'])

print(area_aspirantes)
print(area_carreras)

{1, 2, 3, 4}
{1, 2, 3, 4}


In [ ]:
#@title Variable `carrera`

carrera_aspirantes = set(df_aspirantes['carrera'])
carrera_carreras = set(df_carreras['carrera'])

print("Carreras únicamente en el dataset de aspirantes:", carrera_aspirantes - carrera_carreras)
print("Carreras únicamente en el dataset de carreras:", carrera_carreras - carrera_aspirantes)

Carreras únicamente en el dataset de aspirantes: set()
Carreras únicamente en el dataset de carreras: set()


In [ ]:
#@title Variable `plantel`

plantel_aspirantes = set(df_aspirantes['plantel'])
plantel_carreras = set(df_carreras['plantel'])

print("Planteles únicamente en el dataset de aspirantes:", plantel_aspirantes - plantel_carreras)
print("Planteles únicamente en el dataset de carreras:", plantel_carreras - plantel_aspirantes)

Planteles únicamente en el dataset de aspirantes: set()
Planteles únicamente en el dataset de carreras: set()


Podemos observar que cada una de las variables categóricas es consistente en ambas bases, pues las variables `area` y `concurso` muestran las mismas categorias, al igual que las variables `carrera` y `plantel` contienen el mismo catálogo de valores. De modo que no es necesario realizar procesos adicionales de estandarización sobre estas variables para permitir la integración entre ambas bases.

----

###1.3.2 Consistencia en la información estadística
En esta etapa vamos a analizar la información estadística que almacena cada tabla. Recordemos que la base de datos `estadisticas_aspirantes_UNAM` contiene los registros de cada uno de los aspirantes de ingreso a la licenciatura dentro del periodo 2018 a 2016, mientras que la base `estadisticas_carreras_UNAM` almacena las estadísticas generales de cada carrera en el msmo periodo, es decir nos dice la cantidad de lugares ofertados por carrera, aspirantes, la cantidad de alumnos que si presentaron el examen, los aspirantes seleccionados, entre otros.

Por lo tanto debemos verificar que ambas tablas estén describiendo el mismo comportamiento.

#### Cantidad de aspirantes
Para analizar la consistencia en la cantidad de aspirantes entre ambas bases de datos, se contarán los registros de la base `estadisticas_aspirantes_UNAM` utilizando las variables `año`, `concurso`, `area`, `carrera` y `plantel`, esto para calcular el número de aspirantes mediante el conteo de registros y posteriormente compararlo con el valor reportado en la base `estadisticas_carreras_UNAM`.

In [ ]:
aspirantes_totales = (df_aspirantes.groupby(['año', 'concurso', 'area', 'carrera', 'plantel'], observed=True).size().reset_index(name="aspirantes_totales"))
carreras_aspirantes = (df_carreras.groupby(['año', 'concurso', 'area', 'carrera', 'plantel'], observed=True)['aspirantes'].sum().reset_index(name="aspirantes_carreras"))

comparacion_aspirantes = aspirantes_totales.merge(carreras_aspirantes, on=['año', 'concurso', 'area', 'carrera', 'plantel'], how='outer', validate='one_to_one')
comparacion_aspirantes['diferencia'] = comparacion_aspirantes['aspirantes_totales'] - comparacion_aspirantes['aspirantes_carreras']

comparacion_aspirantes

,año,concurso,area,carrera,plantel,aspirantes_totales,aspirantes_carreras,diferencia
0,2018,Febrero,1,ACTUARIA,FACULTAD DE CIENCIAS,1431,1431,0
1,2018,Febrero,1,ACTUARIA,FES ACATLAN,690,690,0
2,2018,Febrero,1,ARQUITECTURA,FACULTAD DE ARQUITECTURA,3790,3790,0
3,2018,Febrero,1,ARQUITECTURA,FES ACATLAN,1073,1073,0
4,2018,Febrero,1,ARQUITECTURA,FES ARAGON,1366,1366,0
...,...,...,...,...,...,...,...,...
1953,2026,Mayo,4,PEDAGOGIA,FES ARAGON,1591,1591,0
1954,2026,Mayo,4,PIANO,FACULTAD DE MUSICA,30,30,0
1955,2026,Mayo,4,TEATRO Y ACTUACION,FACULTAD DE MUSICA,704,704,0
1956,2026,Mayo,4,TRADUCCION,ESCUELA NAL. DE ESTUDIOS SUP. UNIDAD LEON,29,29,0


In [ ]:
print("Porcentaje de coincidencia exacta:",comparacion_aspirantes["diferencia"].eq(0).mean() * 100, "%")

Porcentaje de coincidencia exacta: 100.0 %


Después de realizar la comparación de aspirantes registrados en ambas tablas, podemos observar que la coincidencia es exacta.

####Cantidad de alumnos que presentaron examen

In [ ]:
presentaron_examen_aspirantes = (df_aspirantes.groupby(['año', 'concurso', 'area', 'carrera', 'plantel'], observed=True))['aciertos'].count().reset_index(name="presentaron_examen_aspirantes")
presentaron_examen_carreras = df_carreras.groupby(['año', 'concurso', 'area', 'carrera', 'plantel'], observed=True)['presentaron_examen'].sum().reset_index(name="presentaron_examen_carreras")

comparacion_presentaron_examen = presentaron_examen_aspirantes.merge(
    presentaron_examen_carreras,
    on=['año', 'concurso', 'area', 'carrera', 'plantel'],
    how='outer',
    validate='one_to_one'
)
comparacion_presentaron_examen['diferencia'] = comparacion_presentaron_examen['presentaron_examen_aspirantes'] - comparacion_presentaron_examen['presentaron_examen_carreras']
comparacion_presentaron_examen

,año,concurso,area,carrera,plantel,presentaron_examen_aspirantes,presentaron_examen_carreras,diferencia
0,2018,Febrero,1,ACTUARIA,FACULTAD DE CIENCIAS,1322,1324,-2
1,2018,Febrero,1,ACTUARIA,FES ACATLAN,641,642,-1
2,2018,Febrero,1,ARQUITECTURA,FACULTAD DE ARQUITECTURA,3506,3516,-10
3,2018,Febrero,1,ARQUITECTURA,FES ACATLAN,991,992,-1
4,2018,Febrero,1,ARQUITECTURA,FES ARAGON,1292,1295,-3
...,...,...,...,...,...,...,...,...
1953,2026,Mayo,4,PEDAGOGIA,FES ARAGON,1352,1386,-34
1954,2026,Mayo,4,PIANO,FACULTAD DE MUSICA,20,21,-1
1955,2026,Mayo,4,TEATRO Y ACTUACION,FACULTAD DE MUSICA,560,576,-16
1956,2026,Mayo,4,TRADUCCION,ESCUELA NAL. DE ESTUDIOS SUP. UNIDAD LEON,14,15,-1


In [ ]:
print("Diferencia total encontrada: ",comparacion_presentaron_examen['diferencia'].abs().sum())

Diferencia total encontrada:  7886


In [ ]:
df_aspirantes = df_aspirantes[
    df_aspirantes["aciertos"].notna()
].copy()

df_aspirantes['aciertos'] = df_aspirantes['aciertos'].astype('int64')
df_aspirantes.shape

(1481624, 8)

Durante la auditoría se identificaron 7,886 registros en los que ambas tablas presentaban diferencias respecto al número de aspirantes. Una investigación en la fuente oficial permitió determinar que estos casos corresponden a aspirantes cuyo proceso de admisión quedó pendiente por una situación administrativa (por ejemplo, "Cita para aclarar situación escolar"), cuyo examen fue cancelado o que no contaban con un resultado válido del examen. Por otro lado, se pudo determinar que existían 223,622 registros sin información dentro de la variable `aciertos`, los cuales incluyen estos casos y otros aspirantes que no presentaron el examen. Debido a que el objetivo del estudio es analizar el comportamiento de los resultados del examen y la evolución de los aciertos mínimos de ingreso, todos los registros sin un puntaje en la variable aciertos fueron excluidos del conjunto de datos de análisis. En consecuencia, la población de estudio quedó conformada únicamente por los aspirantes con un resultado válido del examen.

----
####Cantidad de alumnos sleccionados

In [ ]:
seleccionados_aspirantes = df_aspirantes[df_aspirantes['seleccionado'] == 'Sí'].groupby(['año', 'concurso', 'area', 'carrera', 'plantel'], observed=True).size().reset_index(name="seleccionados_aspirantes")
seleccionados_carreras = df_carreras.groupby(['año', 'concurso', 'area', 'carrera', 'plantel'], observed=True)['seleccionados'].sum().reset_index(name="seleccionados_carreras")

comparacion_seleccionados = seleccionados_aspirantes.merge(
    seleccionados_carreras,
    on=['año', 'concurso', 'area', 'carrera', 'plantel'],
    how='outer',
    validate='one_to_one'
)
comparacion_seleccionados['diferencia'] = comparacion_seleccionados['seleccionados_aspirantes'] - comparacion_seleccionados['seleccionados_carreras']
comparacion_seleccionados.head()

,año,concurso,area,carrera,plantel,seleccionados_aspirantes,seleccionados_carreras,diferencia
0,2018,Febrero,1,ACTUARIA,FACULTAD DE CIENCIAS,41.0,43,-2.0
1,2018,Febrero,1,ACTUARIA,FES ACATLAN,35.0,36,-1.0
2,2018,Febrero,1,ARQUITECTURA,FACULTAD DE ARQUITECTURA,194.0,204,-10.0
3,2018,Febrero,1,ARQUITECTURA,FES ACATLAN,67.0,68,-1.0
4,2018,Febrero,1,ARQUITECTURA,FES ARAGON,60.0,63,-3.0


In [ ]:
print("Diferencia total encontrada: ", comparacion_seleccionados['diferencia'].abs().sum())

Diferencia total encontrada:  4734.0


La comparación del número de seleccionados entre ambas bases mostró una diferencia total de 4,734 aspirantes. La revisión de los registros individuales permitió determinar que esta diferencia corresponde a aspirantes incluidos como seleccionados en las estadísticas oficiales de la UNAM, pero cuyo resultado individual aparece bajo el mensaje “Cita para aclarar situación escolar”. Estos registros no cuentan con un número de aciertos publicado ni con una marca explícita de selección en la base de aspirantes. La cantidad oficial de seleccionados de la tabla de carreras se conservará sin modificaciones; sin embargo, estos aspirantes serán excluidos de los análisis relacionados con puntajes, debido a la ausencia de información en la variable aciertos.

----

##1.4 Dataset final
Después de concluir la auditoría, preparación y validación de datos, ambos conjuntos quedaron listos para iniciar el análisis estadístico. La población de estudio quedó conformada únicamente por aspirantes con un puntaje válido en la variable aciertos, garantizando la consistencia entre ambas fuentes y la calidad de la información utilizada en los análisis posteriores.

In [ ]:
df_aspirantes_final = df_aspirantes.to_parquet('estadisticas_aspirantes_UNAM_corregido.parquet')
df_carreras_final = df_carreras.to_parquet('estadisticas_carreras_UNAM_corregido.parquet')